# ☕ Welcome to Sunny Bay Roastery

## 🎯 Objectives
By the end of this lab, you will:
- Have a **Databricks** workspace (Free Edition works too).
- Have run **one notebook — this Lab 0** — that deploys the whole workshop: your **catalog and schemas** (`bronze`, `silver`, `gold`), the Sunny Bay sales data, the medallion pipeline, the metric view, the pre-built **Sales Genie**, and the **dashboards**.
- Understand the **Sunny Bay Roastery** story and your role in it.

## 🚀 Set Up Your Workspace (One Click)

Setting up is one click: **click `Run all` at the top of this notebook** (serverless — no cluster to pick). Lab 0 creates your catalog, deploys the workshop assets, and **launches** the setup job.

The last cell prints a link to the running job. The job finishes server-side in about **10–15 minutes** — Lab 0 hands it off and releases its compute, which on Free Edition keeps serverless capacity free for the pipeline so the setup runs reliably. When the job shows **Succeeded**, your workspace is ready and you can head to **Lab 1**.

Before you run it, you can optionally change the two parameters below — the defaults work as-is.

In [0]:
# ── Edit these if you need to (the defaults work as-is) ──────────────────
catalog = "serverless_stable_w41qug_catalog"   # the Unity Catalog catalog to build everything in

# Optional. Set this only if several people share the SAME catalog and schema
# (for example "alice"). It is added to every object name — tables, views and
# the metric view — so your objects don't collide with a colleague's. Leave it
# empty ("") if you are working on your own.
prefix = "syrine_"
# ─────────────────────────────────────────────────────────────────────────

# Schema names are fixed by the workshop pipeline — leave these as-is.
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"

In [0]:
import re
from pathlib import Path

# Normalize the prefix: add a trailing underscore so names read cleanly
# (e.g. "alice" -> "alice_dim_customer"). An empty prefix leaves names unchanged.
prefix = prefix.strip()
if prefix and not prefix.endswith("_"):
    prefix += "_"

params = {
    "catalog": catalog,
    "bronze_schema": bronze_schema,
    "silver_schema": silver_schema,
    "gold_schema": gold_schema,
    # Quoted so an empty prefix stays an empty string ("") instead of YAML null.
    "prefix": f'"{prefix}"',
}

path = Path("../bundle/databricks.yml")
text = path.read_text()
for key, value in params.items():
    text = re.sub(rf"^(      {key}: ).*$", lambda m, v=value: m.group(1) + v, text, flags=re.M)
path.write_text(text)

try:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
    print(f"✅ Catalog `{catalog}` is ready.")
except Exception:
    # Some workspaces restrict catalog creation. Fall back to an existing
    # catalog — make sure the `catalog` value above matches a catalog you own.
    spark.sql(f"USE CATALOG {catalog}")
    print(f"⚠️ Could not create catalog `{catalog}` (creation may be restricted "
          f"on this workspace). Using the existing catalog `{catalog}` instead.")

from pathlib import Path

path = Path("../bundle/src/dashboards/dashboard_final.lvdash.json")
text = path.read_text()
text = (
    text.replace('__CATALOG__', catalog)
        .replace('__GOLD_SCHEMA__', gold_schema)
        .replace('__PREFIX__', prefix)
)
path.write_text(text)

import os
import re
import subprocess
import tempfile

# Lab 0 lives in <repo>/labs, so the bundle root (where databricks.yml lives) is one
# directory up, in <repo>/bundle.
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
bundle_root = os.path.abspath("../bundle")
if not os.path.exists(os.path.join(bundle_root, "databricks.yml")):
    raise RuntimeError(
        f"databricks.yml not found at {bundle_root}. Run this notebook from inside the "
        "cloned Git Folder (labs/Lab 0 - Intro), next to the bundle/ tree."
    )

# Install the Databricks CLI. The setup script picks its own bin dir and prints
# "Installed Databricks CLI vX.Y.Z at <path>." — parse that path.
_install_dir = tempfile.mkdtemp(prefix="dbcli_")
_p = subprocess.run(
    ["bash", "-c",
     f"curl -fsSL -m 90 https://raw.githubusercontent.com/databricks/setup-cli/main/install.sh "
     f"| sh -s -- {_install_dir}"],
    capture_output=True, text=True,
)
_m = re.search(r"Installed Databricks CLI \S+ at (\S+?)\.?$", _p.stdout.strip(), re.M)
if _m and os.path.exists(_m.group(1)):
    CLI = _m.group(1)
elif os.path.exists(os.path.join(_install_dir, "databricks")):
    CLI = os.path.join(_install_dir, "databricks")
else:
    _which = subprocess.run(["bash", "-c", "command -v databricks || true"],
                            capture_output=True, text=True).stdout.strip()
    CLI = _which if _which and os.path.exists(_which) else None
if not CLI:
    raise RuntimeError(f"Could not install the Databricks CLI.\nstdout={_p.stdout}\nstderr={_p.stderr}")

# Authenticate with the notebook's own token so deploy/run act as you. Pin the bundle
# root so the CLI finds databricks.yml when we run from the Git Folder.
cli_env = dict(
    os.environ,
    DATABRICKS_HOST=ctx.apiUrl().get(),
    DATABRICKS_TOKEN=ctx.apiToken().get(),
    DATABRICKS_BUNDLE_ROOT=bundle_root,
)


def run_cli(args):
    """Run the CLI streaming combined output into the notebook, raise on failure."""
    print(f"$ databricks {' '.join(args)}")
    proc = subprocess.Popen(
        [CLI, *args], cwd=bundle_root, env=cli_env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"`databricks {' '.join(args)}` failed (exit {proc.returncode}).")


print(subprocess.run([CLI, "--version"], capture_output=True, text=True).stdout.strip())
run_cli(["current-user", "me", "-o", "json"])

# Deploy the bundle: the Sunny Bay Roastery Job, the medallion Lakeflow pipeline, and the
# read-only "[Final]" AI/BI dashboard — all pointed at the catalog/prefix you set above
# (already written into databricks.yml a few cells up).
run_cli(["bundle", "deploy", "-t", "dev", "--force-lock"])
print("✅ Bundle deployed (job, pipeline, and the [Final] dashboard).")

# ── Give yourself an editable dashboard to build in Lab 3 ──────────────────
# The bundle deploys the read-only "[Final]" dashboard as an answer key. The
# branded starter template, though, must be *editable* — a bundle-managed
# dashboard is locked (no Edit button) in the UI. So we create an unmanaged,
# editable copy of the template in your own workspace; Lab 3 builds on this one.
import json as _json


def _cli_json(args):
    _r = subprocess.run([CLI, *args], cwd=bundle_root, env=cli_env,
                        capture_output=True, text=True)
    if _r.returncode != 0:
        raise RuntimeError(f"`databricks {' '.join(args)}` failed: {_r.stderr}")
    return _json.loads(_r.stdout) if _r.stdout.strip() else None


DASH_NAME = "Sunny Bay Roastery - Sales Report"
_me = _cli_json(["current-user", "me", "-o", "json"])["userName"]
_parent = f"/Users/{_me}"

# Reuse the copy if you have run Lab 0 before, so re-runs don't pile up duplicates.
DASH_ID = next(
    (d["dashboard_id"] for d in (_cli_json(["lakeview", "list", "-o", "json"]) or [])
     if d.get("display_name") == DASH_NAME and d.get("lifecycle_state") == "ACTIVE"),
    None,
)
if DASH_ID:
    print(f"✅ Editable dashboard already exists: {DASH_NAME}")
else:
    _whs = _cli_json(["warehouses", "list", "-o", "json"]) or []
    _wh = (next((w for w in _whs if w.get("name") == "Serverless Starter Warehouse"), None)
           or (_whs[0] if _whs else None))
    if not _wh:
        raise RuntimeError("No SQL warehouse available to attach to the dashboard.")
    _tmpl = Path(bundle_root) / "src" / "dashboards" / "dashboard_template.lvdash.json"
    DASH_ID = _cli_json([
        "lakeview", "create",
        "--display-name", DASH_NAME,
        "--warehouse-id", _wh["id"],
        "--dataset-catalog", catalog,
        "--dataset-schema", gold_schema,
        "--serialized-dashboard", _tmpl.read_text(),
        "--json", _json.dumps({"parent_path": _parent}),
        "-o", "json",
    ])["dashboard_id"]
    print(f"✅ Created editable dashboard: {DASH_NAME}")

DASH_URL = f"{ctx.apiUrl().get()}/sql/dashboardsv3/{DASH_ID}"

# Launch the setup job WITHOUT waiting. `--no-wait` returns as soon as the job is
# triggered, so THIS notebook finishes and releases its serverless compute. That
# matters on Free Edition, where serverless capacity is limited: if Lab 0 kept
# running while the job's Lakeflow pipeline started, the two would compete for
# compute and the pipeline could fail with a resource-limit error.
#
# The job then runs server-side (~10-15 min): it generates the sales data, runs the
# bronze -> silver -> gold pipeline, builds the metric view, and pre-builds the
# Sales Genie. You watch it from the Jobs UI (link printed by the next cell).
print("$ databricks bundle run sunny_bay_roastery_job -t dev --restart --no-wait")
_run = subprocess.run(
    [CLI, "bundle", "run", "sunny_bay_roastery_job", "-t", "dev", "--restart", "--no-wait"],
    cwd=bundle_root, env=cli_env, capture_output=True, text=True,
)
print(_run.stdout, end="")
if _run.returncode != 0:
    print(_run.stderr)
    raise RuntimeError(f"Failed to start the setup job (exit {_run.returncode}).")

# Surface the run URL so participants can click straight to the running job.
_m = re.search(r"https?://\S+", _run.stdout)
RUN_URL = _m.group(0).rstrip(".,)") if _m else None

p = prefix or ""
print("=" * 68)
print("🚀  SETUP STARTED  —  your workshop job is now running server-side")
print("=" * 68)
print()
if RUN_URL:
    print("Watch progress here:")
    print(f"  {RUN_URL}")
else:
    print("Watch progress in the left sidebar under Jobs & Pipelines →")
    print('  the run named "[dev <you>] Sunny Bay Roastery Job".')
print()
print("It takes about 10-15 minutes. This notebook has finished and released its")
print("compute on purpose: on Free Edition that frees serverless capacity for the")
print("pipeline, so the setup runs reliably.")
print()
print(f"When the job succeeds, everything will be in catalog `{catalog}`:")
print(f"  🥉 Bronze raw    : {catalog}.{bronze_schema}.raw (Volume) + generated CSV/Parquet")
print(f"  🥈 Silver / 🥇 Gold star schema : {catalog}.{gold_schema}.{p}fact_coffee_sales + {p}dim_*")
print(f"  📐 Metric view   : {catalog}.{gold_schema}.{p}sm_fact_coffee_sales_genie")
print(f"  🧞 Sales Genie   : pre-built over the metric view")
print(f'  📊 Dashboard     : "{DASH_NAME}" (editable — you build this in Lab 3)')
print(f'                     plus a read-only "[Final]" version as the answer key')
print(f"  ✏️  Edit your dashboard: {DASH_URL}")
print()
print("Once the job shows Succeeded, open  labs/Lab 1  and follow along!")

## ☕ Story Setup

In 2013, the very same year **Databricks** was founded, a small team of coffee enthusiasts opened a café in **San Francisco**. They called it **Sunny Bay Roastery**.

At first, they were just another boutique coffee shop, but their obsession with precision, data, and quality soon made them a local favorite. Every espresso shot was logged.

Over the next decade, Sunny Bay grew to five stores across the Bay Area. Then, in 2020, when the pandemic hit, foot traffic dropped overnight. The company had to act fast.

Online coffee bean sales exploded as people became **home baristas**, experimenting with grinders and brewing ratios while stuck at home.

Today, in 2025, Sunny Bay Roastery stands at a crossroads.  
Its CEO, **Mr. Bean**, wants to understand:
> "What really drives our coffee sales?  
>  How do seasons, holidays, and online vs. in-store trends shape our future?"

Unfortunately, the company's data is scattered:
- Some comes from old **in-store point-of-sale systems**.  
- Some from **e-commerce logs**.  
- And some information are in **Excel files** sitting in Mr. Bean's inbox.

You've just been hired as the company's first **Head of Data & Analytics**.  
Your mission: **build a unified data platform** that can turn these fragments into insight — and prepare Sunny Bay Roastery for its next phase of growth.